# D3PM on 32x32 Mazes

A categorical discrete denoising diffusion model for binary mazes. Each pixel remains a class (`0 = wall`, `1 = passage`) throughout the forward and reverse Markov chains. The forward process uses uniform categorical transitions, and a timestep-conditioned U-Net predicts the clean pixel classes. The learned reverse categorical transitions are sampled at every step.

Run cells manually; training may take a while.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import Denoiser

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project: {PROJECT_ROOT} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA runtime: {torch.version.cuda}")
else:
    print(
        f"CUDA unavailable to PyTorch (torch.version.cuda={torch.version.cuda}). Run uv sync and select the project .venv kernel."
    )

Project: c:\Users\shich\Src\discrete_diffusion | device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | CUDA runtime: 12.8


## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. `0` means wall; `1` means passage.

In [2]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()
print(
    "all:", mazes.shape, "| train:", len(train_mazes), "| validation:", len(val_mazes)
)
print(f"Generated-maze solvability: {solvability_rate(mazes):.1%}")

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

all: (12000, 32, 32) | train: 10800 | validation: 1200
Generated-maze solvability: 100.0%


## 2. Categorical Forward Process

For two classes, use the uniform transition matrix

`Q_t = gamma_t I + (1 - gamma_t) U`, where `U` has every entry equal to `1/2`.

The cumulative transition is `Q_bar_t = alpha_bar_t I + (1 - alpha_bar_t) U`. Thus each pixel is still exactly `0` or `1`, and as `alpha_bar_t` approaches zero its distribution approaches a fair coin. We sample `x_t` directly from this cumulative distribution, rather than simulating every forward step.

In [3]:
NUM_CLASSES = 2
DIFFUSION_STEPS = 100


def make_d3pm_matrices(num_steps, device, offset=0.008):
    times = torch.linspace(0, 1, num_steps + 1, device=device)
    cumulative = torch.cos(((times + offset) / (1 + offset)) * torch.pi / 2).square()
    alpha_bars = (cumulative / cumulative[0]).clamp_min(1e-5)
    alpha_bars[0] = 1.0

    identity = torch.eye(NUM_CLASSES, device=device)
    uniform = torch.full((NUM_CLASSES, NUM_CLASSES), 1.0 / NUM_CLASSES, device=device)
    transition_matrices = torch.empty(
        num_steps + 1, NUM_CLASSES, NUM_CLASSES, device=device
    )
    transition_matrices[0] = identity
    for step in range(1, num_steps + 1):
        gamma = alpha_bars[step] / alpha_bars[step - 1]
        transition_matrices[step] = gamma * identity + (1.0 - gamma) * uniform

    cumulative_matrices = (
        alpha_bars[:, None, None] * identity
        + (1.0 - alpha_bars[:, None, None]) * uniform
    )
    return alpha_bars, transition_matrices, cumulative_matrices


alpha_bars, transition_matrices, cumulative_matrices = make_d3pm_matrices(
    DIFFUSION_STEPS, DEVICE
)


def q_sample(x0, timesteps):
    keep_probability = (1.0 + alpha_bars[timesteps]) / 2.0
    flip = torch.rand(x0.shape, device=x0.device) >= keep_probability[:, None, None]
    return torch.bitwise_xor(x0, flip.long())


def one_hot_pixels(x):
    return F.one_hot(x.long(), num_classes=NUM_CLASSES).permute(0, 3, 1, 2).float()


preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
preview_noise_steps = [
    0,
    1,
    2,
    4,
    7,
    11,
    16,
    22,
    30,
    39,
    49,
    59,
    69,
    79,
    89,
    94,
    97,
    100,
]
with torch.no_grad():
    preview_frames = [preview_maze[0].cpu().numpy()]
    for step in preview_noise_steps[1:]:
        t = torch.full((1,), step, device=DEVICE, dtype=torch.long)
        preview_frames.append(q_sample(preview_maze, t)[0].cpu().numpy())

forward_fig = make_subplots(
    rows=2,
    cols=9,
    subplot_titles=[f"t={step}" for step in preview_noise_steps],
)
for index, frame in enumerate(preview_frames):
    forward_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=index // 9 + 1,
        col=index % 9 + 1,
    )
forward_fig.update_layout(
    title="D3PM forward corruption (binary states at every step)",
    width=1600,
    height=650,
)
forward_fig.update_xaxes(showticklabels=False)
forward_fig.update_yaxes(showticklabels=False, autorange="reversed")
forward_fig.show()

print(f"alpha_bar_T={alpha_bars[-1].item():.6f}")
print("Final cumulative transition matrix:")
print(cumulative_matrices[-1].detach().cpu())

alpha_bar_T=0.000010
Final cumulative transition matrix:
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])


## 3. Model Architecture

The same U-Net layout as the other notebooks receives a two-channel one-hot image and a discrete timestep. Its two output channels are logits for the clean class `x_0` at each pixel, not noise or velocity. The reverse transition distribution is computed analytically from these clean-class probabilities and the known D3PM posterior.

In [4]:
BASE_CHANNELS = 64
TIME_DIM = 128

model = Denoiser(
    embedding_dim=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
).to(DEVICE)
print(model)
print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Denoiser(
  (time_mlp): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): SiLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (input): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc32): ResBlock(
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=64, bias=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (skip): Identity()
  )
  (down16): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (enc16): ResBlock(
    (norm1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=128, bias=True)
    (norm2): GroupNorm(8, 128, eps=1e-05, a

## 4. D3PM Training

Sample `t` uniformly from `1..T`, corrupt the maze with the exact categorical marginal `q(x_t | x_0)`, and predict `p_theta(x_0 | x_t)`. For `t > 1`, train the implied reverse categorical distribution against the exact posterior `q(x_{t-1} | x_t, x_0)` using KL divergence. At `t = 1`, use the reconstruction cross-entropy. A small auxiliary clean-class cross-entropy encourages accurate `x_0` predictions at every timestep. The live Plotly loss chart updates in place.

In [5]:
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 100
AUXILIARY_X0_WEIGHT = 0.001

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def make_posterior_lookup():
    lookup = torch.zeros(
        DIFFUSION_STEPS + 1,
        NUM_CLASSES,
        NUM_CLASSES,
        NUM_CLASSES,
        device=DEVICE,
    )
    for t in range(1, DIFFUSION_STEPS + 1):
        q_t = transition_matrices[t]
        q_bar_previous = cumulative_matrices[t - 1]
        q_bar_current = cumulative_matrices[t]
        for x_t in range(NUM_CLASSES):
            for x0 in range(NUM_CLASSES):
                weights = q_t[:, x_t] * q_bar_previous[x0, :]
                lookup[t, x_t, x0, :] = weights / q_bar_current[x0, x_t]
    return lookup


posterior_lookup = make_posterior_lookup()


def reverse_distribution_from_logits(logits_x0, x_t, timesteps):
    p_x0 = logits_x0.softmax(dim=1)
    x_t_one_hot = one_hot_pixels(x_t)
    posterior_for_t = posterior_lookup[timesteps]
    posterior_for_pixels = torch.einsum(
        "bcuv,bchw->bhwuv", posterior_for_t, x_t_one_hot
    )
    p_previous = torch.einsum("buhw,bhwuv->bvhw", p_x0, posterior_for_pixels)
    return p_x0, p_previous.clamp_min(1e-12)


def batch_loss(maze_batch):
    x0 = maze_batch.to(DEVICE)
    timesteps = torch.randint(1, DIFFUSION_STEPS + 1, (x0.shape[0],), device=DEVICE)
    x_t = q_sample(x0, timesteps)
    logits_x0 = model(one_hot_pixels(x_t), timesteps)
    p_x0, p_previous = reverse_distribution_from_logits(logits_x0, x_t, timesteps)

    x0_one_hot = one_hot_pixels(x0)
    posterior_for_pixels = torch.einsum(
        "bcuv,bchw->bhwuv", posterior_lookup[timesteps], one_hot_pixels(x_t)
    )
    true_previous = torch.einsum("buhw,bhwuv->bvhw", x0_one_hot, posterior_for_pixels)
    pixel_kl = (
        true_previous * (true_previous.clamp_min(1e-12).log() - p_previous.log())
    ).sum(dim=1)
    kl_per_example = pixel_kl.mean(dim=(1, 2))
    x0_ce_per_example = F.cross_entropy(logits_x0, x0, reduction="none").mean(
        dim=(1, 2)
    )
    vb_or_reconstruction = torch.where(
        timesteps == 1, x0_ce_per_example, kl_per_example
    )
    return (vb_or_reconstruction + AUXILIARY_X0_WEIGHT * x0_ce_per_example).mean()


@torch.no_grad()
def validation_loss():
    model.eval()
    values = [batch_loss(batch).item() for (batch,) in val_loader]
    model.train()
    return float(np.mean(values))


model.train()
train_history, val_history = [], []
loss_fig = go.Figure()
loss_fig.add_scatter(name="train", mode="lines")
loss_fig.add_scatter(name="validation", mode="lines+markers")
loss_fig.update_layout(
    title="D3PM variational and auxiliary loss",
    xaxis_title="Optimization step",
    yaxis_title="Loss per maze",
)
global_step = 0
for epoch in range(EPOCHS):
    for (batch,) in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(batch)
        loss.backward()
        optimizer.step()
        train_history.append(loss.item())
        global_step += 1
        if global_step % PLOT_EVERY == 0:
            loss_fig.data[0].x = list(range(1, len(train_history) + 1))
            loss_fig.data[0].y = train_history
            loss_fig.data[1].x = [step for step, _ in val_history]
            loss_fig.data[1].y = [value for _, value in val_history]
            loss_fig.update_layout(
                title=f"D3PM training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
            )
            clear_output(wait=True)
            display(loss_fig)
    epoch_train = float(np.mean(train_history[-len(train_loader) :]))
    epoch_validation = validation_loss()
    val_history.append((global_step, epoch_validation))
    loss_fig.data[0].x = list(range(1, len(train_history) + 1))
    loss_fig.data[0].y = train_history
    loss_fig.data[1].x = [step for step, _ in val_history]
    loss_fig.data[1].y = [value for _, value in val_history]
    loss_fig.update_layout(
        title=f"D3PM training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
    )
    clear_output(wait=True)
    display(loss_fig)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | train={epoch_train:.4f} | val={epoch_validation:.4f}"
    )

print(f"Training complete | train={epoch_train:.4f} | val={epoch_validation:.4f}")

Epoch 30/30 | train=0.0017 | val=0.0017
Training complete | train=0.0017 | val=0.0017


## 5. Save the Model

In [6]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_d3pm_uniform_x0.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "num_classes": NUM_CLASSES,
            "diffusion_steps": DIFFUSION_STEPS,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "transition": "uniform_binary_categorical",
            "prediction_target": "x0_logits",
            "auxiliary_x0_weight": AUXILIARY_X0_WEIGHT,
        },
    },
    checkpoint_path,
)
print(f"Saved: {checkpoint_path}")

Saved: c:\Users\shich\Src\discrete_diffusion\artifacts\maze_d3pm_uniform_x0.pt


## 6. Reverse Sampling and Solvability

Start from the terminal prior, which is an independent fair binary state at every pixel. At each reverse timestep, the U-Net predicts clean-class probabilities; combine these with the exact categorical posterior to get `p_theta(x_{t-1} | x_t)`, then sample each pixel categorically. Unlike the continuous notebooks, the entire reverse trajectory consists of binary grids.

In [7]:
N_METRIC_SAMPLES = 128
N_TO_SHOW = 4
REVERSE_PREVIEW_TIMESTEPS = [100, 90, 80, 70, 60, 50, 40, 30, 20, 10, 0]


@torch.no_grad()
def sample_d3pm(model, n_samples, capture_timesteps=()):
    model.eval()
    x_t = torch.randint(
        0, NUM_CLASSES, (n_samples, 32, 32), device=DEVICE, dtype=torch.long
    )
    initial_prior = x_t.detach().clone()
    captures = {DIFFUSION_STEPS: x_t.detach().clone()}
    capture_timesteps = set(capture_timesteps)

    for t in reversed(range(1, DIFFUSION_STEPS + 1)):
        timesteps = torch.full((n_samples,), t, device=DEVICE, dtype=torch.long)
        logits_x0 = model(one_hot_pixels(x_t), timesteps)
        _, p_previous = reverse_distribution_from_logits(logits_x0, x_t, timesteps)
        probabilities = p_previous.permute(0, 2, 3, 1).reshape(-1, NUM_CLASSES)
        x_t = torch.multinomial(probabilities, num_samples=1).reshape(n_samples, 32, 32)
        if (t - 1) in capture_timesteps:
            captures[t - 1] = x_t.detach().clone()

    return x_t, captures, initial_prior


samples, reverse_snapshots, initial_prior = sample_d3pm(
    model, N_METRIC_SAMPLES, REVERSE_PREVIEW_TIMESTEPS
)
hard_samples = samples.cpu().numpy().astype(np.uint8)
print(
    "Terminal prior class-1 fraction: "
    f"{initial_prior.float().mean().item():.3f} (expected about 0.5)"
)

fig = make_subplots(
    rows=N_TO_SHOW,
    cols=len(REVERSE_PREVIEW_TIMESTEPS),
    subplot_titles=[
        f"t={t}" for _ in range(N_TO_SHOW) for t in REVERSE_PREVIEW_TIMESTEPS
    ],
)
for row in range(N_TO_SHOW):
    for col, timestep in enumerate(REVERSE_PREVIEW_TIMESTEPS, start=1):
        fig.add_trace(
            go.Heatmap(
                z=reverse_snapshots[timestep][row].cpu().numpy(),
                zmin=0,
                zmax=1,
                colorscale=[[0, "black"], [1, "white"]],
                showscale=False,
            ),
            row=row + 1,
            col=col,
        )
fig.update_layout(
    title="Reverse D3PM sampling: categorical maze state at each timestep",
    height=850,
    width=1900,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()

solved = [solvability_rate([maze]) for maze in hard_samples]
print(f"Solvable samples: {sum(solved)}/{len(solved)} ({np.mean(solved):.1%})")
hard_fig = make_subplots(
    rows=1,
    cols=N_TO_SHOW,
    subplot_titles=[
        f"Sample {i + 1}: {'solvable' if solved[i] else 'unsolvable'}"
        for i in range(N_TO_SHOW)
    ],
)
for i in range(N_TO_SHOW):
    hard_fig.add_trace(
        go.Heatmap(
            z=hard_samples[i],
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=i + 1,
    )
hard_fig.update_layout(title="Final D3PM maze samples", height=390, width=900)
hard_fig.update_xaxes(showticklabels=False)
hard_fig.update_yaxes(showticklabels=False, autorange="reversed")
hard_fig.show()

Terminal prior class-1 fraction: 0.501 (expected about 0.5)


Solvable samples: 62.0/128 (48.4%)
